<a href="https://colab.research.google.com/github/paulaprado1904/AlgoritmoEvolutivo/blob/main/AE_nH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import time
import numpy as np
import pandas as pd
import os

# Contador global de avaliações completas da função objetivo.
# É usado para limitar o esforço computacional total do algoritmo.
GLOBAL_AVALIACOES = 0


# ======================================================
# === Estrutura do indivíduo (representação absoluta) ===
# ======================================================
class Individuo:
    """
    Representa uma solução candidata para o problema de dobramento proteico
    no modelo HP em 3D.

    Cada indivíduo é codificado por uma sequência de movimentos absolutos
    em uma malha cúbica tridimensional. A partir dessa sequência, são
    reconstruídas as coordenadas da cadeia e calculadas as métricas de avaliação.
    """
    def __init__(self, movimentos):
        # Cromossomo do indivíduo: sequência de movimentos absolutos
        self.movimentos = movimentos[:]

        # Coordenadas reconstruídas da cadeia na malha
        self.coords = None

        # Número de colisões (sobreposição de resíduos)
        self.nC = 0

        # Número de contatos H-H não consecutivos
        self.nH = 0

        # Valor da função objetivo principal
        self.f = None

        # Métricas auxiliares de compactação
        self.dmax = None
        self.davg = None

        # Campo mantido por compatibilidade com outras variantes
        self.delta = None

        # Energia equivalente da solução
        self.energia = None

    def __str__(self):
        """
        Retorna uma representação textual resumida do indivíduo.
        Útil para inspeção manual e depuração.
        """
        f_val = self.f if self.f is not None else 0
        d_max = self.dmax if self.dmax is not None else 0
        d_avg = self.davg if self.davg is not None else 0
        delta_val = self.delta if self.delta is not None else 0
        energia_val = self.energia if self.energia is not None else 0

        return (
            f"f(c): {f_val:3} | "
            f"nC: {self.nC:2} | "
            f"dmax: {d_max:5.2f} | "
            f"davg: {d_avg:5.2f} | "
            f"delta: {delta_val:6.2f} | "
            f"Energia: {energia_val}"
        )


# ======================================================
# === Conversão de movimento absoluto para deslocamento ===
# ======================================================
def delta_from_move_3d(mov):
    """
    Converte o código de um movimento em seu deslocamento cartesiano
    correspondente na malha 3D.

    Convenção adotada:
    0 -> +x
    1 -> +y
    2 -> +z
    3 -> -z
    4 -> -y
    5 -> -x
    """
    if mov == 0:   # U: +x
        return (1, 0, 0)
    if mov == 1:   # L: +y
        return (0, 1, 0)
    if mov == 2:   # F: +z
        return (0, 0, 1)
    if mov == 3:   # B: -z
        return (0, 0, -1)
    if mov == 4:   # R: -y
        return (0, -1, 0)
    if mov == 5:   # D: -x
        return (-1, 0, 0)
    raise ValueError("mov inválido (esperado 0..5)")


# ======================================================
# === Construção das coordenadas e contagem de colisões ===
# ======================================================
def construir_coordenadas(ind):
    """
    Reconstrói as coordenadas da proteína a partir da sequência de movimentos
    do indivíduo.

    Retorna:
    - coords: lista de coordenadas ocupadas pela cadeia
    - nC: número de colisões detectadas

    Uma colisão ocorre quando dois ou mais resíduos ocupam a mesma posição
    na malha.
    """
    x = y = z = 0
    coords = [(0, 0, 0)]
    ocupacao = {(0, 0, 0): 1}

    for mov in ind.movimentos:
        dx, dy, dz = delta_from_move_3d(mov)
        x += dx
        y += dy
        z += dz
        pt = (x, y, z)
        coords.append(pt)
        ocupacao[pt] = ocupacao.get(pt, 0) + 1

    nC = sum(1 for v in ocupacao.values() if v > 1)

    ind.coords = coords
    ind.nC = nC
    return coords, nC


def avaliar_base(ind, cadeia, rho=1.0):
    """
    Avaliação completa do indivíduo.

    Métricas calculadas:
    - nC: número de colisões
    - nH: número de contatos hidrofóbicos H-H não consecutivos
    - f(c): função objetivo principal
    - dmax: maior distância entre resíduos H
    - davg: distância média entre resíduos H
    - energia: energia equivalente usada no resumo dos resultados

    Regras:
    - Se houver colisão, a solução é tratada como inviável.
    - Soluções inviáveis recebem penalização na função objetivo.
    - Neste algoritmo, o foco principal está na maximização de nH.
    """
    global GLOBAL_AVALIACOES
    GLOBAL_AVALIACOES += 1

    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    # Salva as coordenadas no indivíduo para possível reutilização
    ind.coords = coords

    # --------------------------------------------------
    # Cálculo das distâncias entre resíduos H
    # --------------------------------------------------
    H_idx = [i for i, a in enumerate(cadeia) if a == "H"]
    dists = []
    m = len(H_idx)

    for a in range(m):
        for b in range(a + 1, m):
            p1 = coords[H_idx[a]]
            p2 = coords[H_idx[b]]

            # Ignora pares que colapsaram para a mesma posição devido a colisão
            if p1 == p2:
                continue

            d = math.sqrt(
                (p1[0] - p2[0]) ** 2 +
                (p1[1] - p2[1]) ** 2 +
                (p1[2] - p2[2]) ** 2
            )
            dists.append(d)

    if not dists:
        dmax = float('inf')
        davg = float('inf')
    else:
        dmax = max(dists)
        davg = sum(dists) / len(dists)

    # --------------------------------------------------
    # Caso inviável: há colisões
    # --------------------------------------------------
    if nC > 0:
        ind.nC = nC
        ind.nH = 0
        ind.f = -rho * nC

        # Mantém dmax e davg com penalização para não perder totalmente
        # a informação estrutural da solução
        P = 10.0 * nC
        ind.dmax = dmax + P
        ind.davg = davg + P

        ind.delta = None
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Caso viável: sem colisões
    # Conta contatos H-H não consecutivos e adjacentes no espaço
    # --------------------------------------------------
    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):  # ignora adjacentes na cadeia
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]

            # Distância Manhattan entre os resíduos i e j
            manh = (
                abs(pi[0] - pj[0]) +
                abs(pi[1] - pj[1]) +
                abs(pi[2] - pj[2])
            )

            # Se a distância Manhattan for 1, há contato espacial
            if manh == 1:
                nH += 1

    ind.nC = 0
    ind.nH = nH
    ind.f = nH
    ind.dmax = dmax
    ind.davg = davg
    ind.delta = None
    ind.energia = -ind.f


# ======================================================
# === Geração de indivíduo aleatório ===
# ======================================================
def gerar_individuo_aleatorio(n_residuos):
    """
    Gera um indivíduo aleatório com n_residuos - 1 movimentos.

    Nesta variante, todos os 6 movimentos possíveis podem aparecer
    na geração inicial.
    """
    L = n_residuos - 1
    movimentos = [random.randint(0, 5) for _ in range(L)]
    return Individuo(movimentos)


# ======================================================
# === Recombinação por múltiplos pontos ===
# ======================================================
def crossover_multipontos_2filhos(pai, mae, n_residuos):
    """
    Gera dois filhos a partir de crossover de múltiplos pontos.

    O número de cortes cresce proporcionalmente ao tamanho da proteína.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    num_cortes = max(1, min(num_cortes, L - 1))
    cortes = sorted(random.sample(range(1, L), num_cortes))

    def build_child(start_with_pai: bool):
        filho_mov = []
        pais = [pai.movimentos, mae.movimentos]
        idx = 0 if start_with_pai else 1
        ini = 0
        for c in cortes + [L]:
            filho_mov.extend(pais[idx][ini:c])
            idx = 1 - idx
            ini = c
        return Individuo(filho_mov)

    return build_child(True), build_child(False)


# ======================================================
# === Mutação de dois genes consecutivos ===
# ======================================================
def mutacao_dois_genes(ind, taxa_mut):
    """
    Aplica mutação em dois genes consecutivos com probabilidade taxa_mut.

    Se a mutação ocorrer, escolhe-se uma posição aleatória e os dois genes
    consecutivos são redefinidos com novos movimentos aleatórios.
    """
    novo = Individuo(ind.movimentos[:])

    if random.random() < taxa_mut:
        L = len(novo.movimentos)
        if L >= 2:
            pos = random.randint(0, L - 2)
            for k in (pos, pos + 1):
                novo.movimentos[k] = random.randint(0, 5)

    return novo


def selecao_torneio_mono(pop, k=3):
    """
    Seleção por torneio monoobjetivo.

    O critério de escolha é a maximização do fitness f.
    """
    if not pop:
        raise RuntimeError("População vazia no torneio.")

    cand = random.sample(pop, k=min(k, len(pop)))
    return max(cand, key=lambda ind: ind.f)


def selecao_sobreviventes_mu_plus_lambda(pop, filhos, pop_size, elitismo=1):
    """
    Seleção de sobreviventes no esquema (mu + lambda).

    Estratégia:
    - preserva um número fixo de indivíduos elite da população anterior;
    - combina pais e filhos;
    - remove clones idênticos por cromossomo, favorecendo diversidade;
    - preenche a nova população com os melhores indivíduos restantes.
    """
    # Garante elitismo: melhores pais entram diretamente
    elite = sorted(pop, key=lambda ind: ind.f, reverse=True)[:max(0, elitismo)]

    candidatos = pop + filhos

    # Remove clones idênticos por cromossomo
    vistos = set()
    unicos = []
    for ind in sorted(candidatos, key=lambda ind: ind.f, reverse=True):
        key = tuple(ind.movimentos)
        if key not in vistos:
            vistos.add(key)
            unicos.append(ind)

    # Força a entrada do elite no topo da nova população
    nova = []
    elite_keys = {tuple(e.movimentos) for e in elite}
    for e in elite:
        nova.append(e)

    for ind in unicos:
        if len(nova) >= pop_size:
            break
        if tuple(ind.movimentos) in elite_keys:
            continue
        nova.append(ind)

    # Se ainda faltar indivíduo, completa sem filtro
    while len(nova) < pop_size:
        nova.append(random.choice(candidatos))

    return nova


def aemo_hp_3d_sem_mc(
    cadeia_raw,
    pop_init=8,
    taxa_mut=0.2,
    rho=1.0,
    torneio_k=2,
    taxa_cross=1.0,
    max_avaliacoes=None,
    seed=None,
    elitismo=1,
    target_nH=None
):
    """
    Executa o algoritmo evolutivo monoobjetivo voltado à maximização de nH.

    Estrutura geral:
    1. Gera população inicial aleatória
    2. Avalia todos os indivíduos
    3. Seleciona pais por torneio
    4. Gera filhos por crossover e mutação
    5. Aplica seleção de sobreviventes no esquema (mu + lambda)
    6. Atualiza estatísticas até atingir o critério de parada

    Critérios de parada possíveis:
    - atingir o número máximo de avaliações
    - atingir um valor alvo de nH (target_nH)
    """
    global GLOBAL_AVALIACOES

    cadeia = ''.join([c for c in cadeia_raw.strip().upper() if c in ('H', 'P')])
    n_res = len(cadeia)
    if n_res < 2:
        raise ValueError("Cadeia muito curta")

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ---------------- População inicial ----------------
    populacao = []
    for _ in range(pop_init):
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break

        ind = gerar_individuo_aleatorio(n_res)
        avaliar_base(ind, cadeia, rho=rho)
        populacao.append(ind)

    if not populacao:
        raise RuntimeError("Nenhum indivíduo pôde ser avaliado (max_avaliacoes muito baixo).")

    # Melhor indivíduo global até o momento
    melhor_global = max(populacao, key=lambda ind: ind.f)
    geracao_melhor = 0
    aval_ate_target = None

    # Histórico por geração
    melhores_f = []
    medias_f = []
    desvios_f = []

    geracao = 0
    while True:
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break

        if target_nH is not None and melhor_global.nC == 0 and melhor_global.nH >= target_nH:
            if aval_ate_target is None:
                aval_ate_target = GLOBAL_AVALIACOES
            break

        # ---------- Geração de filhos ----------
        filhos = []
        while len(filhos) < pop_init:
            pai = selecao_torneio_mono(populacao, k=torneio_k)
            mae = selecao_torneio_mono(populacao, k=torneio_k)

            if random.random() < taxa_cross:
                f1, f2 = crossover_multipontos_2filhos(pai, mae, n_res)
            else:
                f1, f2 = Individuo(pai.movimentos[:]), Individuo(mae.movimentos[:])

            for filho in (f1, f2):
                if len(filhos) >= pop_init:
                    break

                filho = mutacao_dois_genes(filho, taxa_mut)
                avaliar_base(filho, cadeia, rho=rho)
                filhos.append(filho)

                print(f"Individuo gerado filho: {filho}")

        # ---------- Seleção de sobreviventes ----------
        populacao = selecao_sobreviventes_mu_plus_lambda(
            populacao,
            filhos,
            pop_size=pop_init,
            elitismo=elitismo
        )

        # ---------- Estatísticas da geração ----------
        f_vals = np.array([ind.f for ind in populacao], dtype=float)
        medias_f.append(float(np.mean(f_vals)))
        desvios_f.append(float(np.std(f_vals)))

        melhor_ger = max(populacao, key=lambda ind: ind.f)
        melhores_f.append(melhor_ger.f)

        if melhor_ger.f > melhor_global.f:
            melhor_global = melhor_ger
            geracao_melhor = geracao + 1

        geracao += 1

    resumo = {
        "Melhor Fitness (nH penalizado se colisão)": melhor_global.f,
        "nH(c)": melhor_global.nH,
        "nC(c)": melhor_global.nC,
        "Geração do Melhor": geracao_melhor,
        "Aval_ate_target": aval_ate_target,
        "Aval_total": GLOBAL_AVALIACOES,
    }

    df_hist = pd.DataFrame({
        "Iteracao": list(range(len(melhores_f))),
        "Fitness Médio": medias_f,
        "Desvio Padrão": desvios_f,
        "Melhor Fitness": melhores_f
    })

    return df_hist, resumo, melhor_global


# ======================================================
# === Execução repetida dos experimentos ===
# ======================================================
if __name__ == "__main__":
    # Arquivo com a instância da proteína e pasta de saída
    caminho_arquivo = "/content/drive/MyDrive/pastaOrigem/a3.txt"
    pasta_saida = "/content/drive/MyDrive/pastaDestino/resultados AEMO nH"
    os.makedirs(pasta_saida, exist_ok=True)

    # Leitura do arquivo de entrada:
    # linha 1 -> tamanho da cadeia
    # linha 2 -> sequência HP
    with open(caminho_arquivo) as f:
        n = int(f.readline().strip())
        cadeia = ''.join([c for c in f.readline().strip().upper() if c in ('H', 'P')])
        assert len(cadeia) == n

    nome_proteina = os.path.splitext(os.path.basename(caminho_arquivo))[0]

    # Ótimo conhecido da instância
    ENERGIA_OTIMA = -8

    if n <= 36:
        # Parâmetros experimentais para proteínas pequenas
        taxa_mut = 0.03
        taxa_cross = 0.80
        MAX_AVALIACOES = 500_000

    NUM_EXECUCOES = 50
    resultados_execucoes = []

    for i in range(NUM_EXECUCOES):
        GLOBAL_AVALIACOES = 0
        random.seed()
        np.random.seed()

        print(f"\n========== EXECUÇÃO {i+1}/{NUM_EXECUCOES} ==========")

        # Como energia = -nH em soluções factíveis, o ótimo em nH
        # é o oposto da energia ótima conhecida
        NH_OTIMO = -ENERGIA_OTIMA

        df_hist, resumo, melhor = aemo_hp_3d_sem_mc(
            cadeia,
            pop_init=120,
            taxa_mut=taxa_mut,
            rho=1.0,
            torneio_k=3,
            taxa_cross=taxa_cross,
            max_avaliacoes=MAX_AVALIACOES,
            elitismo=0,
            target_nH=NH_OTIMO,
        )

        # Energia equivalente só faz sentido para solução factível
        melhor_energia = (-melhor.nH) if melhor.nC == 0 else float("inf")

        hit_otimo = (
            (resumo["Aval_ate_target"] is not None) and
            (melhor.nC == 0) and
            (melhor.nH >= NH_OTIMO)
        )
        aval_ate_otimo = resumo["Aval_ate_target"]

        resultados_execucoes.append({
            "Execucao": i + 1,
            "Melhor_Energia": melhor_energia,
            "Hit_otimo": hit_otimo,
            "Aval_ate_otimo": aval_ate_otimo,
            "Aval_total": resumo["Aval_total"],
            "Melhor_fitness": resumo["Melhor Fitness (nH penalizado se colisão)"],
            "nH": melhor.nH,
            "nC": melhor.nC,
        })

    df_resumo = pd.DataFrame(resultados_execucoes)

    # ======================================================
    # === Estatísticas da melhor energia por execução ===
    # ======================================================
    energies = df_resumo["Melhor_Energia"].replace([np.inf, -np.inf], np.nan).dropna()

    melhor_energia_media = float(energies.mean()) if not energies.empty else float("nan")
    melhor_energia_dp = float(energies.std(ddof=1)) if len(energies) > 1 else float("nan")

    qtd_facteis = int(energies.shape[0])
    qtd_total = int(df_resumo.shape[0])

    print("\n\n===== RESUMO DAS 50 EXECUÇÕES =====")
    print(df_resumo)

    # Considera apenas execuções que atingiram o ótimo conhecido
    df_sucesso = df_resumo[df_resumo["Hit_otimo"] == True]

    if not df_sucesso.empty:
        media_aval = df_sucesso["Aval_ate_otimo"].mean()
        mediana_aval = df_sucesso["Aval_ate_otimo"].median()
    else:
        media_aval = float('nan')
        mediana_aval = float('nan')

    taxa_acertos = 100.0 * df_sucesso.shape[0] / NUM_EXECUCOES

    print("\n===== RESUMO GLOBAL (estilo artigo) =====")
    print(f"Ótimo conhecido: {ENERGIA_OTIMA}")
    print(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}")
    print(f"Acurácia (Ac.): {taxa_acertos:.1f} %")
    print(f"Média Aval. (apenas execuções com ótimo): {media_aval:.0f}")
    print(f"Mediana Aval.: {mediana_aval:.0f}")

    # Salva resumo final em arquivo texto
    caminho_saida_txt = os.path.join(pasta_saida, f"{nome_proteina}.txt")
    with open(caminho_saida_txt, "w") as f_out:
        f_out.write(f"Arquivo da proteína: {caminho_arquivo}\n")
        f_out.write(f"Nome da proteína (base): {nome_proteina}\n")
        f_out.write(f"Ótimo conhecido: {ENERGIA_OTIMA}\n\n")

        f_out.write("===== RESUMO DAS 50 EXECUÇÕES =====\n")
        f_out.write(df_resumo.to_string(index=False))
        f_out.write("\n\n===== RESUMO GLOBAL (estilo artigo) =====\n")
        f_out.write(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}\n")
        f_out.write(f"Acurácia (Ac.): {taxa_acertos:.1f} %\n")
        f_out.write(f"Média Aval. (execuções com ótimo): {media_aval:.0f}\n")
        f_out.write(f"Mediana Aval.: {mediana_aval:.0f}\n")

        f_out.write("\n----- ESTATÍSTICAS (MELHOR ENERGIA POR EXECUÇÃO) -----\n")
        f_out.write(f"Melhor_Energia (média ± dp): {melhor_energia_media:.4f} ± {melhor_energia_dp:.4f}\n")
        f_out.write(f"Execuções factíveis consideradas: {qtd_facteis}/{qtd_total}\n")

    print(f"\nResumo salvo em: {caminho_saida_txt}")